In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import os
import sys

import cv2

import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F

from box import Box
import yaml

import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as R
import random 

import time 

root_dir = os.path.abspath('../..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

# from src.models.dpso_inference import DPSO

from src.data_loader.evaluation_data_generator import DataGenerator
from src.models.patchifier import Patchifier
# from training.utils import pose_err


In [13]:
model_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'
sonar_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/sonar.yaml'

data_root_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/train/seq_1'
data_output_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/training/test_output'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(model_config_pth, "r") as f:
            model_config = Box(yaml.safe_load(f))

with open(sonar_config_pth, "r") as f:
            sonar_config = Box(yaml.safe_load(f))

# init_frames = model_config.TIME_WINDOW

from src.data_loader.transforms import SonarDatasetTranforms
data_generator = DataGenerator(data_root_dir, device, transforms = SonarDatasetTranforms)

from src.data_loader.utils import img_polar2cart

data_lenght = data_generator.get_len()
print(f'Data generator initialized.')
print(f'Data series lenght: {data_lenght}')

patchifier = Patchifier(model_config)

Data generator initialized.
Data series lenght: 500


In [16]:
# --- harris ---

limit_frames = 30 
r_min = sonar_config.range.min
r_max = sonar_config.range.max
theta_max = sonar_config.fov.horizontal

wait_time_ms = 200 

window_name = 'Sonar Tracking'
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 800, 400)

with torch.no_grad():
    for frame_idx in range(limit_frames):

        t, frame, pose, frame_np = data_generator.get_sample(frame_idx, return_visu=True)
        
        coords, patches_f, patches_c, fmap = patchifier(frame, mode='harris')

        frame_np_visu, harris_response = patchifier.get_visu(frame, coords)
        
        frame_keypoints = img_polar2cart(frame_np_visu, r_min, r_max, theta_max, out_shape=None, bg=0)


        cv2.imshow(window_name, frame_keypoints)
        

        key = cv2.waitKey(wait_time_ms) & 0xFF
        if key == ord('q'):
            break

cv2.destroyAllWindows()

In [17]:
# --- DoG ---

limit_frames = 30 
r_min = sonar_config.range.min
r_max = sonar_config.range.max
theta_max = sonar_config.fov.horizontal

wait_time_ms = 200 

window_name = 'Sonar Tracking'
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 800, 400)

with torch.no_grad():
    for frame_idx in range(limit_frames):

        t, frame, pose, frame_np = data_generator.get_sample(frame_idx, return_visu=True)
        
        coords, patches_f, patches_c, fmap = patchifier(frame, mode='DoG')

        frame_np_visu, harris_response = patchifier.get_visu(frame, coords)
        
        frame_keypoints = img_polar2cart(frame_np_visu, r_min, r_max, theta_max, out_shape=None, bg=0)


        cv2.imshow(window_name, frame_keypoints)
        

        key = cv2.waitKey(wait_time_ms) & 0xFF
        if key == ord('q'):
            break

cv2.destroyAllWindows()

In [18]:
# --- hessian ---

limit_frames = 30 
r_min = sonar_config.range.min
r_max = sonar_config.range.max
theta_max = sonar_config.fov.horizontal

wait_time_ms = 200 

window_name = 'Sonar Tracking'
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 800, 400)

with torch.no_grad():
    for frame_idx in range(limit_frames):

        t, frame, pose, frame_np = data_generator.get_sample(frame_idx, return_visu=True)
        
        coords, patches_f, patches_c, fmap = patchifier(frame, mode='hessian')

        frame_np_visu, harris_response = patchifier.get_visu(frame, coords)
        
        frame_keypoints = img_polar2cart(frame_np_visu, r_min, r_max, theta_max, out_shape=None, bg=0)


        cv2.imshow(window_name, frame_keypoints)
        

        key = cv2.waitKey(wait_time_ms) & 0xFF
        if key == ord('q'):
            break

cv2.destroyAllWindows()